# Monte Carlo EM (MCEM) for a Logistic Random-Intercept GLMM (Wei & Tanner, 1990)

## Background & Motivation

Generalized linear mixed models (GLMMs) introduce **random effects** to account for dependence / heterogeneity across clusters.
A classic example is a **binomial logistic model** with herd-level random intercepts:

- Herds $i=1,\dots,G$, periods $t=1,\dots,T$.
- Observed: $y_{it}$ = number of new cases out of $n_{it}$ tested (so $y_{it}\sim \text{Binomial}(n_{it}, p_{it})$).
- Random intercept: $b_i \stackrel{iid}{\sim} \mathcal{N}(0,\sigma^2)$.

Model:
$$
y_{it}\mid b_i,\beta \sim \text{Binomial}(n_{it}, p_{it}), \qquad
\text{logit}(p_{it}) = x_{it}^\top\beta + b_i.
$$

### Why MCEM?

The observed-data likelihood requires integrating out the random effects:
$$
L(\beta,\sigma^2\mid y) = \prod_{i=1}^{G}\int \left[\prod_{t=1}^{T} f(y_{it}\mid b_i,\beta)\right]\phi(b_i;0,\sigma^2)\,db_i,
$$
which has **no closed form** for logistic GLMMs.

EM would treat $b=(b_1,\dots,b_G)$ as missing data and iterate
$$
Q(\theta\mid\theta^{(t)})=\mathbb{E}_{b\mid y,\theta^{(t)}}[\log f(y,b\mid\theta)].
$$
When this expectation is intractable, **MCEM** (Wei & Tanner, 1990) approximates it using Monte Carlo draws
$$
Q(\theta\mid\theta^{(t)}) \approx \frac{1}{m}\sum_{j=1}^{m}\log f\big(y,b^{(j)}\mid\theta\big),
\quad b^{(j)}\sim p(b\mid y,\theta^{(t)}).
$$

Because of Monte Carlo error, the usual EM ascent property may be lost unless $m$ is large enough (often increased with iteration).

## Real data application: CBPP (Contagious Bovine Pleuropneumonia)

We'll use the `cbpp` dataset (from the R package **lme4**) containing quarterly serological incidence in 15 herds.  
We model the effect of **period** (fixed) and allow **herd-specific baseline risk** via the random intercept $b_i$.

In [1]:
import numpy as np
import pandas as pd
from io import StringIO

# CBPP dataset (lme4::cbpp), embedded for offline use; also try to download a fresh copy.
CBPP_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/cbpp.csv"
df = pd.read_csv(CBPP_URL)

In [2]:

df = df.drop(columns=["rownames"])
df["period"] = df["period"].astype(int)
df["herd"] = df["herd"].astype(int)

# Response and binomial totals
y = df["incidence"].to_numpy(dtype=float)
n_trials = df["size"].to_numpy(dtype=float)

# Fixed effects: intercept + period dummies (baseline period=1)
period = df["period"].to_numpy()
X = np.column_stack([
    np.ones(len(df)),
    (period == 2).astype(float),
    (period == 3).astype(float),
    (period == 4).astype(float),
])

# Herd index 0..G-1
herd_ids = df["herd"].to_numpy()
unique_herds = np.sort(np.unique(herd_ids))
herd_map = {h:i for i,h in enumerate(unique_herds)}
group = np.array([herd_map[h] for h in herd_ids], dtype=int)
G = len(unique_herds)

print(df.head())
print("n_obs =", len(df), "| G =", G, "| p =", X.shape[1])

   herd  incidence  size  period
0     1          2    14       1
1     1          3    12       2
2     1          4     9       3
3     1          0     5       4
4     2          3    22       1
n_obs = 56 | G = 15 | p = 4


## Tasks

### Part A — Theory / Derivation with Explanation

Let $\theta=(\beta,\sigma^2)$ and $b=(b_1,\dots,b_G)$.

1. **Complete-data log-likelihood.**  
   Write $\log f(y,b\mid\beta,\sigma^2)$ up to additive constants.  
   (Hint: include the binomial log-likelihood and the normal prior for $b_i$.)

2. **Conditional density of a random effect.**  
   Show that (up to proportionality) the conditional posterior of each $b_i$ is
   $$
   p(b_i\mid y_i,\beta,\sigma^2)\propto
   \exp\left\{\sum_{t\in \mathcal{T}(i)}\Big[y_{it}(\eta_{it}+b_i)-n_{it}\log(1+e^{\eta_{it}+b_i})\Big]\right\}
   \exp\left(-\frac{b_i^2}{2\sigma^2}\right),
   $$
   where $\eta_{it}=x_{it}^\top\beta$ and $\mathcal{T}(i)$ are the rows belonging to herd $i$.

3. **M-step updates (from Monte Carlo draws).**
   - Show that the maximizer of the Gaussian prior part yields the closed-form update
     $$
     \sigma^2_{\text{new}} = \frac{1}{G}\,\mathbb{E}\Big[\sum_{i=1}^G b_i^2\ \big|\ y,\theta^{(t)}\Big]
     \approx \frac{1}{Gm}\sum_{j=1}^m\sum_{i=1}^G \big(b_i^{(j)}\big)^2.
     $$
   - Derive the score and Hessian for $\beta$ (conditional on $b$), and propose a Newton update.  
     (Hint: for binomial logistic, $p=\text{logit}^{-1}(\eta+b)$ and the Hessian involves $n\,p(1-p)$.)

4. Briefly explain why **MCEM may not be monotone** in the observed-data log-likelihood.

---

### Part B — Implementation / Real data

1. Implement a 1D random-walk Metropolis–Hastings sampler for each $b_i$ (they are conditionally independent across herds).
2. Implement MCEM with:
   - E-step: draw $b^{(1)},\dots,b^{(m)}$ from $p(b\mid y,\theta^{(t)})$
   - M-step: update $\sigma^2$ (closed form) and $\beta$ (Newton)
3. Fit the model and interpret:
   - period fixed effects (do odds increase/decrease over time?)
   - the estimated random-intercept variance $\sigma^2$
4. Compare to a fixed-effects logistic regression (no random intercept).

---

### Part C
- Increase $m$ over EM iterations and see if the parameter trace stabilizes.

## References

- Wei, G. C. G. & Tanner, M. A. (1990). *A Monte Carlo implementation of the EM algorithm and the poor man's data augmentation algorithms*. **JASA**, 85, 699–704.
- CBPP dataset documentation (lme4): Contagious bovine pleuropneumonia incidence in 15 herds.